# GISLR — Curated-Feature DNN + LSTM (ME-126 + xy + joint angles)

**What this notebook does.** Trains a memory-free per-frame **DNN** and a
causal **LSTM** on a deliberately **curated** feature set — not the full-543
landmarks `landmark_interp_v1` uses, but the three winning conclusions from
this session's prior interpretability experiments put together into one
pipeline (`sb.recognize.interp.features_curated`, `landmark_curated_v1`):

1. **ME-126 landmark subset** (`docs/reports/motion-energy.md` +
   `docs/reports/subset-comparison.md` + `docs/reports/landmark-importance.md`
   — three independent methods all picked the same 126 of 543 landmarks:
   lips + hands + eyes/nose + upper-body pose).
2. **xy only, z dropped** (TODO §3.1 — the established default; z is mostly
   noise for pose/most landmarks).
3. **28 engineered joint angles** (`docs/reports/feature-discriminability.md`
   — the single most information-dense feature type found this session: 56
   angle features alone reached 68.3% probe accuracy vs 79.4% for 3,258
   position features).

Result: **922** features per frame, an 83% reduction from `landmark_interp_v1`'s
5,442 — see `features_curated.py`'s module docstring for the exact channel
layout and why z-dropping and angle-computation don't conflict (angles are
computed from the full 3D geometry *before* z is dropped from the raw
coordinate channel).

**Output contract**: both models produce a confidence score (softmax
probability) for **every** of the 250 signs, not just the winner —
`LandmarkDNN.forward` already does this per-frame; `LandmarkRNN.forward_all`
(new) applies the same trained head to every timestep instead of only the
last one. Both are used here as a **video-level** headline metric (matching
this repo's canonical-eval convention) plus a **per-frame diagnostic** (§6).

**Top-N evaluation**: the headline metric isn't "did rank 1 match" but "is
the true label anywhere in the top N" — e.g. if a video's true label is
`awake` and the model's top-1 is `wake` (0.9) with `awake` second (0.8),
that counts as a **top-2 hit**, not a miss. This directly follows up
`docs/reports/pair-similarity.md`/`plateau-diagnosis.md` §3's finding that
confused near-synonyms are usually near-misses, not far misses.

**Pipeline stage vs. diagnostic.** Same status as `landmark-importance.ipynb`
and `feature-discriminability.ipynb`: self-contained under
`sb.recognize.interp`, not part of `sb.recognize.architectures.ARCHS` or the
registry — the point here is testing whether the curated feature recipe
actually trains a good, evaluable model, not adding a leaderboard entry.

**Artifacts**

| Path | Content |
|---|---|
| `data/cache/gislr/features/landmark_curated_v1/<key>/` | curated feature cache (train + test), content-addressed |
| `data/cache/gislr/curated_feature_runs/<arch>/final.pt` | final-fit checkpoint (gitignored) |
| `data/cache/gislr/curated_feature_runs/<arch>/test_predictions.npz` | per-video probs/labels on `test.csv` |
| `data/cache/gislr/curated_feature_runs/topn_summary.csv` | top-1/3/5 accuracy per architecture |
| `docs/reports/curated-features.md` | write-up, **after** this notebook runs |

**Design decisions**

- **Single final fit, not k-fold OOF** — unlike `landmark-importance.ipynb`
  (which needed full out-of-fold coverage to rank every landmark), this
  experiment only needs one good model per architecture: `train_test_split`
  a small internal-val carve-out from `train.csv` for early-stopping signal
  only, then **one** evaluation pass on the untouched canonical `test.csv` —
  same "final fit" pattern §5 of `landmark-importance.ipynb` already used,
  just without the k-fold phase before it.
- **DNN + LSTM only** (no GRU) — per the experiment request.
- Uses a **separate run root** (`curated_feature_runs/`, not
  `interp_runs/`) so this doesn't collide with the already-completed
  landmark-importance checkpoints, which also use `dnn`/`lstm` as directory
  names.


In [ ]:
# ============================================================
# Imports, tunables, resolved paths
# ============================================================
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

from sb.core.paths import CACHE_DIR, EXPERIMENTS_DIR
from sb.recognize.interp import features_curated as CFEAT
from sb.recognize.interp.geometry import RELATIONAL_DIM
from sb.recognize.interp.models import LandmarkDNN, LandmarkRNN
from sb.recognize.interp.train import (
    load_split_arrays,
    make_fold_loader,
    make_row_tracked_loader,
    predict_probs_indexed,
    train_fold,
)
from sb.recognize.sources import get_source

CONFIG_PATH = EXPERIMENTS_DIR / "recognition" / "configs" / "gislr.curated-features.json"
CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))

SEED = CONFIG["seed"]
HYP = CONFIG["shared"]  # every training hyperparameter comes from the config, never this cell
ARCH_CFG = CONFIG["architectures"]
FINAL_VAL_FRACTION = CONFIG["final_fit_val_fraction"]
TOP_N = CONFIG["top_n"]
ARCHITECTURES = ["dnn", "lstm"]

# separate from interp_runs/ (landmark-importance's root) -- distinct arch dirnames would collide otherwise
RUN_ROOT = CACHE_DIR / "gislr" / "curated_feature_runs"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
for arch in ARCHITECTURES:
    (RUN_ROOT / arch / "assets").mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"config: {CONFIG_PATH}")
print(f"device: {DEVICE}")
print(f"run artifacts: {RUN_ROOT}")
print(f"feature_dim: {CFEAT.FEATURE_DIM} (= {CFEAT.PER_LANDMARK_DIM} per-landmark "
      f"+ {CFEAT.ANGLE_DIM} angle + {RELATIONAL_DIM} relational)")
print(f"epochs cap={HYP['epochs']} · batch_size={HYP['batch_size']} · top_n={TOP_N}")


## 1. Curated feature cache

Builds `sb.recognize.interp.features_curated`'s flat `(T, 922)`-per-frame
cache for `train.csv` and `test.csv` — content-addressed under
`data/cache/gislr/features/landmark_curated_v1/<key>/`, skip-if-exists like
every other pipeline cache in this repo.

In [ ]:
# ============================================================
# Canonical split + curated-feature caches (train.csv, test.csv)
# ============================================================
ds = get_source("gislr")
DATA_DIR = ds.resolve_dir()
sign2idx = ds.label_map(DATA_DIR)
idx2sign = {v: k for k, v in sign2idx.items()}
N_CLASSES = len(sign2idx)

train_split, test_split = ds.canonical_split(DATA_DIR, sign2idx)
print(f"train.csv: {len(train_split)} videos - test.csv: {len(test_split)} videos - {N_CLASSES} classes")

def _progress(done, total, label):
    if done == 0:
        print(f"{label}: caching {total} videos (this is the slow step, run once)...")

tr_data_path, tr_off_path = CFEAT.build_cache(
    train_split, "train", DATA_DIR, progress=lambda d, t: _progress(d, t, "train")
)
te_data_path, te_off_path = CFEAT.build_cache(
    test_split, "test", DATA_DIR, progress=lambda d, t: _progress(d, t, "test")
)
print(f"cache dir: {CFEAT.cache_dir(DATA_DIR)}")


## 2. Model factories

In [ ]:
# ============================================================
# Model factories -- every architecture-specific hyperparameter from CONFIG,
# every dimension from the curated pipeline
# ============================================================
CURATED_DIMS = dict(
    n_landmarks=CFEAT.N_LANDMARKS,
    channels_per_landmark=CFEAT.CHANNELS_PER_LANDMARK,
    angle_dim=CFEAT.ANGLE_DIM,
    relational_dim=RELATIONAL_DIM,
)


def make_model(arch: str):
    cfg = ARCH_CFG[arch]
    if arch == "dnn":
        return LandmarkDNN(
            hidden_sizes=tuple(cfg["hidden_sizes"]), num_classes=N_CLASSES, dropout=cfg["dropout"],
            **CURATED_DIMS,
        )
    return LandmarkRNN(
        cell=arch, proj_size=cfg["proj_size"], hidden_size=cfg["hidden_size"],
        num_layers=cfg["num_layers"], num_classes=N_CLASSES, dropout=cfg["dropout"],
        **CURATED_DIMS,
    )


n_params = {arch: sum(p.numel() for p in make_model(arch).parameters()) for arch in ARCHITECTURES}
pd.Series(n_params, name="n_params").to_frame()


## 3. Final fit (one model per architecture, trained on (nearly) all of `train.csv`)

No k-fold here — this experiment doesn't need out-of-fold coverage, only one
good model per architecture. The internal-val carve-out drives early
stopping/LR only and is never a reported metric (same convention
`landmark-importance.ipynb` §5 used).

In [ ]:
# ============================================================
# Final fit: one model per architecture
# ============================================================
train_data, train_off, train_labels = load_split_arrays(tr_data_path, tr_off_path, train_split)

tr_idx_final, val_idx_final = train_test_split(
    np.arange(len(train_split)), test_size=FINAL_VAL_FRACTION,
    stratify=train_split["label"], random_state=SEED,
)
print(f"final fit: {len(tr_idx_final)} train / {len(val_idx_final)} internal-val "
      f"(early-stopping signal only -- never a reported metric)")

final_models = {}
for arch in ARCHITECTURES:
    ckpt = RUN_ROOT / arch / "final.pt"
    tr_loader = make_fold_loader(train_data, train_off, train_labels, tr_idx_final,
                                  HYP["batch_size"], True, SEED, feature_dim=CFEAT.FEATURE_DIM)
    va_loader = make_fold_loader(train_data, train_off, train_labels, val_idx_final,
                                  HYP["batch_size"], False, SEED, feature_dim=CFEAT.FEATURE_DIM)
    bar = tqdm(total=HYP["epochs"], desc=f"{arch} - final fit", dynamic_ncols=True)
    model, history, best_val_acc, best_state = train_fold(
        arch=arch, model_fn=lambda a=arch: make_model(a), train_loader=tr_loader, val_loader=va_loader,
        hyp=HYP, device=DEVICE, ckpt_path=ckpt, bar=bar, bar_prefix="final",
    )
    bar.close()
    model.load_state_dict(best_state)
    model.eval()
    final_models[arch] = model
    (RUN_ROOT / arch / "history.json").write_text(json.dumps(history))
    print(f"{arch}: final fit best internal-val acc {best_val_acc:.4f}")


## 4. Held-out test-set evaluation + top-N accuracy

One pass per model over the untouched canonical `test.csv`. Top-N accuracy =
the true label is anywhere in the N highest-confidence predictions, for
N in `TOP_N` (config-driven, default [1, 3, 5]) — the same "ranked
alternatives" concept `sb.recognize.evaluate.py` stores for the canonical
leaderboard (`TOPK=5`, `topk_idx`/`topk_prob`), applied here as an explicit
accuracy metric rather than just stored alternatives.

In [ ]:
# ============================================================
# ONE evaluation pass per final model, on the untouched canonical test.csv
# ============================================================
test_data, test_off, test_labels_arr = load_split_arrays(te_data_path, te_off_path, test_split)

test_results = {}
topn_rows = []
for arch in ARCHITECTURES:
    loader = make_row_tracked_loader(
        test_data, test_off, test_labels_arr, np.arange(len(test_split)), HYP["batch_size"],
        feature_dim=CFEAT.FEATURE_DIM,
    )
    rows, probs, ret_labels = predict_probs_indexed(final_models[arch], arch, loader, DEVICE)
    order = np.argsort(rows)
    labels_ordered, probs_ordered = ret_labels[order], probs[order]

    ranked = np.argsort(-probs_ordered, axis=-1)  # (n_videos, N_CLASSES), best-first
    for n in TOP_N:
        hit = (ranked[:, :n] == labels_ordered[:, None]).any(axis=1)
        topn_rows.append({"arch": arch, "top_n": n, "accuracy": float(hit.mean())})

    test_results[arch] = dict(labels=labels_ordered, probs=probs_ordered, ranked=ranked)
    np.savez(RUN_ROOT / arch / "test_predictions.npz",
             labels=labels_ordered, probs=probs_ordered, ranked_top5=ranked[:, :5])

topn_df = pd.DataFrame(topn_rows).pivot(index="arch", columns="top_n", values="accuracy")
topn_df.columns = [f"top_{n}_acc" for n in topn_df.columns]
topn_df.to_csv(RUN_ROOT / "topn_summary.csv")
topn_df


## 5. Confusable-pair check: does top-N recover the known near-synonyms?

`docs/reports/pair-similarity.md`/`plateau-diagnosis.md` documented 16
semantically-confused sign pairs (`awake`/`wake`, `lips`/`mouth`, ...). For
each pair present in this run's test set, show how often the *wrong* member
of the pair wins rank 1 while the *correct* one is recovered at rank 2 —
concretely the scenario the user described (`wake` at 0.9, `awake` at 0.8,
still a top-2 hit).

In [ ]:
# ============================================================
# Confusable-pair rank-recovery, per architecture
# ============================================================
CONFUSABLE_PAIRS = [
    ("awake", "wake"), ("lips", "mouth"), ("hear", "listen"), ("pen", "pencil"),
    ("gift", "give"), ("cut", "scissors"), ("finger", "wait"), ("stay", "that"),
    ("duck", "goose"), ("animal", "have"), ("cat", "kitty"), ("sleep", "sleepy"),
    ("nap", "sleep"), ("bed", "bedroom"), ("dry", "dryer"), ("look", "see"),
]

for arch in ARCHITECTURES:
    labels_ordered = test_results[arch]["labels"]
    ranked = test_results[arch]["ranked"]
    print(f"\n=== {arch} ===")
    rows = []
    for a, b in CONFUSABLE_PAIRS:
        if a not in sign2idx or b not in sign2idx:
            continue
        for true_sign, other_sign in ((a, b), (b, a)):
            true_idx, other_idx = sign2idx[true_sign], sign2idx[other_sign]
            mask = labels_ordered == true_idx
            if mask.sum() == 0:
                continue
            r = ranked[mask]
            top1 = (r[:, 0] == true_idx).mean()
            top1_is_other = (r[:, 0] == other_idx).mean()
            top2_recovers = ((r[:, 0] == other_idx) & (r[:, 1] == true_idx)).sum()
            rows.append({
                "true_sign": true_sign, "confused_with": other_sign, "n": int(mask.sum()),
                "top1_acc": top1, "rank1_is_other": top1_is_other,
                "recovered_at_rank2": int(top2_recovers),
            })
    print(pd.DataFrame(rows).to_string(index=False))


## 6. Per-frame confidence trace (diagnostic, not the headline metric)

`LandmarkDNN.forward` and `LandmarkRNN.forward_all` both produce a
confidence score for every label at every frame. This section plots that for
a few example test videos — the true label's confidence and its top
competitor's, frame by frame — as concrete evidence for whether confidence
in the right answer grows as more of the sign is seen. Read honestly (see
`sb.recognize.interp.models.LandmarkRNN.forward_all`'s docstring): the LSTM
was trained with a loss at the last frame only, so early-frame confidence was
never supervised to be meaningful — this is a diagnostic, and a visual seed
for the backlogged live-streaming idea (`TODO.md` "Backlog / Someday"), not a
claim that early frames are already reliable.

In [ ]:
# ============================================================
# Per-frame confidence traces for a handful of example test videos
# ============================================================
@torch.no_grad()
def frame_confidences(arch, row):
    d = test_off[row + 1] - test_off[row]
    feat_dim = CFEAT.FEATURE_DIM
    arr = test_data[test_off[row] * feat_dim: test_off[row + 1] * feat_dim].reshape(-1, feat_dim)
    x = torch.from_numpy(np.ascontiguousarray(arr)).unsqueeze(0).to(DEVICE)
    lengths = torch.tensor([arr.shape[0]])
    model = final_models[arch]
    if arch == "dnn":
        logits = model(x)  # (1, T, C)
    else:
        logits = model.forward_all(x, lengths)  # (1, T, C)
    return torch.softmax(logits[0], dim=-1).cpu().numpy()  # (T, C)


EXAMPLE_SIGNS = ["awake", "scissors"]  # one confusable-pair example, one control
example_rows = [
    int(test_split.index[test_split["sign"] == s][0]) for s in EXAMPLE_SIGNS if s in set(test_split["sign"])
]
if not example_rows:  # EXAMPLE_SIGNS not present in this run's class set (e.g. a shrunk smoke test) -- fall back
    example_rows = [0, min(1, len(test_split) - 1)]
    print(f"EXAMPLE_SIGNS not found in this run; falling back to rows {example_rows} "
          f"({[test_split.iloc[r]['sign'] for r in example_rows]})")

fig, axes = plt.subplots(len(example_rows), len(ARCHITECTURES),
                          figsize=(6 * len(ARCHITECTURES), 3.5 * len(example_rows)), squeeze=False)
for i, row in enumerate(example_rows):
    true_sign = test_split.iloc[row]["sign"]
    true_idx = sign2idx[true_sign]
    for j, arch in enumerate(ARCHITECTURES):
        probs = frame_confidences(arch, row)  # (T, C)
        top_competitor = np.argsort(-probs[-1])[1] if np.argsort(-probs[-1])[0] == true_idx else np.argsort(-probs[-1])[0]
        ax = axes[i, j]
        ax.plot(probs[:, true_idx], label=f"true: {true_sign}")
        ax.plot(probs[:, top_competitor], label=f"competitor: {idx2sign[top_competitor]}", linestyle="--")
        ax.set_ylim(0, 1)
        ax.set_title(f"{arch} - {true_sign}")
        ax.set_xlabel("frame")
        ax.set_ylabel("confidence")
        ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


## 7. Summary & artifact index

In [ ]:
# ============================================================
# Summary
# ============================================================
print("Top-N accuracy:")
print(topn_df)
print()
print("Params:", n_params)
print()
print("Artifacts:")
for arch in ARCHITECTURES:
    print(f"  {RUN_ROOT / arch / 'final.pt'}")
    print(f"  {RUN_ROOT / arch / 'test_predictions.npz'}")
print(f"  {RUN_ROOT / 'topn_summary.csv'}")
